In [1]:
import os
from collections import defaultdict
from itertools import chain

import polars as pl
from social_groups.analysis.defs.notebooks.definitions import register_materialization

from social_groups.analysis.polars_transformations import make_group_constellation
from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
    SingularityVote,
)
from social_groups.reporting.parsing import (
    AnswerOptions,
    AnswerParser,
    AnswerComparer,
)
from social_groups.directories import REPORTING_DIR
from social_groups.polars_values import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.group_decision_scheme import calculate_decision_scheme

from social_groups.analysis.polars_transformations.apply_parsing_and_group_decision import (
    apply_parsing_and_group_decision,
)

%load_ext autoreload
%autoreload 2

In [2]:
output_dir = REPORTING_DIR / "heterogeneous_group"
os.makedirs(output_dir, exist_ok=True)

triple_underscore_handling = "wrong"

In [3]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling=triple_underscore_handling
)

## Analzying Basic Group Behaviour

In [4]:
from social_groups.analysis.definitions import defs

mad_frame = defs().load_asset_value("hetero_mad")
baseline_frame = defs().load_asset_value("baseline")
original_table_page46 = defs().load_asset_value(["report", "external", "group_problem_solving_page_46"])

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_config/pythonic_config/typing_utils.py:111: UserWarning: Field name "extension" in "PolarsParquetIOManager" shadows an attribute in parent "BasePolarsUPathIOManager"
  return super().__new__(cls, name, bases, namespaces, **kwargs)
2026-03-10 12:39:42 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/hetero_mad.parquet using PolarsParquetIOManager...
2026-03-10 12:39:43 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/baseline.parquet using PolarsParquetIOManager...
2026-03-10 12:39:43 +0800 - dagster - DEBUG - system - Loading file from: /var/folders/f0/f9gys8xx0psgwxr0dzsfp95h0000gn/T/tmpwyxemh93/storage/report/external/group_problem_solving_page_46 using PickledObjectFilesystemIOMan

FileNotFoundError: [Errno 2] No such file or directory: '/var/folders/f0/f9gys8xx0psgwxr0dzsfp95h0000gn/T/tmpwyxemh93/storage/report/external/group_problem_solving_page_46'

In [6]:
mad_frame.head()

shape: (5, 17)
┌──────┬────────┬─────────────┬─────────────┬───┬──────────┬─────────────┬────────────┬────────────┐
│ id   ┆ run_id ┆ question_id ┆ phoenix_spa ┆ … ┆ category ┆ question    ┆ answer_str ┆ model_name │
│ ---  ┆ ---    ┆ ---         ┆ n_id        ┆   ┆ ---      ┆ ---         ┆ ing        ┆ s          │
│ i64  ┆ i64    ┆ i64         ┆ ---         ┆   ┆ str      ┆ str         ┆ ---        ┆ ---        │
│      ┆        ┆             ┆ str         ┆   ┆          ┆             ┆ str        ┆ list[str]  │
╞══════╪════════╪═════════════╪═════════════╪═══╪══════════╪═════════════╪════════════╪════════════╡
│ 3260 ┆ 43     ┆ 5           ┆ 1aa14e4c915 ┆ … ┆ health   ┆ Q: What     ┆ J          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ cb1a3       ┆   ┆          ┆ stable      ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆             ┆   ┆          ┆ isotope is  ┆            ┆            │
│      ┆        ┆             ┆             ┆   ┆          ┆ comm…       ┆            ┆            │
│ 3261 ┆ 43     ┆ 6           ┆ c0ea9a68165 ┆ … ┆ physics  ┆ Q: An       ┆ I          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ ac33e       ┆   ┆          ┆ electric    ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆             ┆   ┆          ┆ dipole      ┆            ┆            │
│      ┆        ┆             ┆             ┆   ┆          ┆ consisti…   ┆            ┆            │
│ 3262 ┆ 43     ┆ 0           ┆ 6bc64c3c961 ┆ … ┆ other    ┆ Q: In 2018, ┆ G          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ beda7       ┆   ┆          ┆ about how   ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆             ┆   ┆          ┆ many chi…   ┆            ┆            │
│ 3263 ┆ 43     ┆ 8           ┆ 35ee8080254 ┆ … ┆ law      ┆ Q: A        ┆ E          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 89b27       ┆   ┆          ┆ company     ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆             ┆   ┆          ┆ contracted  ┆            ┆            │
│      ┆        ┆             ┆             ┆   ┆          ┆ with a…     ┆            ┆            │
│ 3264 ┆ 43     ┆ 2           ┆ ee9c12870a3 ┆ … ┆ physics  ┆ Q: Kirkwood ┆ F          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ b979e       ┆   ┆          ┆ gaps are    ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆             ┆   ┆          ┆ observed …  ┆            ┆            │
└──────┴────────┴─────────────┴─────────────┴───┴──────────┴─────────────┴────────────┴────────────┘

In [7]:
# TODO: Is this mixed for different datasets? (MAD-> subset, Baseline -> Full Eval?)
baseline_analysis = (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .with_columns(
        is_correct=comparer(
            pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
        )
    )
    .group_by("model_name")
    .agg(accuracy=pl.col("is_correct").mean())
)

In [8]:
model_name_sort = {
    "Qwen/Qwen3-14B": 1,
    "Qwen/Qwen3-4B": 2,
    "Qwen/Qwen3-0.6B": 3,
}

mad_analysis = apply_parsing_and_group_decision(
    mad_frame, parser, comparer, group_reply
).with_columns(make_group_constellation())

table_page_46 = pl.concat(
    [
        (
            mad_analysis.group_by("group_constellation")
            .agg(accuracy=pl.col("is_correct").mean())
            .with_columns(origin=pl.lit("(MAD)"))
        ),
        (
            baseline_analysis.with_columns(
                group_constellation=pl.col("model_name").replace(
                    MODEL_NAME_TO_LETTER_MAPPING
                ),
            ).with_columns(origin=pl.lit("(baseline)"))
        ),
    ],
    how="diagonal",
)

register_materialization("heterogeneous_groups_evaluation_table", table_page_46, "The main evaluation of different group constellations for Multi Agent Debate, can be nicely compared to the table on page 46 of Group Problem Solving Book.")
table_page_46

group_constellation,accuracy,origin,model_name
str,f64,str,str
"""M""",0.55,"""(MAD)""",null
"""HL""",0.53,"""(MAD)""",null
"""LLLM""",0.61,"""(MAD)""",null
"""LL""",0.28,"""(MAD)""",null
"""HM""",0.65,"""(MAD)""",null
…,…,…,…
"""HLL""",0.46,"""(MAD)""",null
"""HHHM""",0.67,"""(MAD)""",null
"""H""",0.64,"""(baseline)""","""Qwen/Qwen3-14B"""


In [9]:
original_table_page46

group_constellation,score (-115 to 115)
str,i64
"""HHH""",80
"""HHM""",74
"""HHL""",67
"""HML""",64
"""HMM""",61
…,…
"""M""",42
"""MML""",39
"""MLL""",37


In [10]:
decision_schemes = (
    mad_analysis.group_by("group_constellation")
    .map_groups(
        lambda g: calculate_decision_scheme(
            g,
            AnalysisColumn.parsed_individual_answers_before.value,
            AnalysisColumn.parsed_individual_answers_after.value,
            "answer_string",
            group_reply,
            comparer,
        ).select(
            pl.lit(g["group_constellation"].unique().item()).alias(
                "group_constellation"
            ),
            "Correct Members Beginning",
            "correct",
            "incorrect",
        )
    )
    .sort("group_constellation")
)

decision_schemes.write_csv(output_dir / "group_decision_schemes.csv")

decision_schemes

group_constellation,Correct Members Beginning,correct,incorrect
str,u32,f64,f64
"""H""",1,0.986111,0.013889
"""H""",0,0.142857,0.857143
"""HH""",2,1.0,0.0
"""HH""",1,0.6875,0.3125
"""HH""",0,0.181818,0.818182
…,…,…,…
"""MMMM""",4,0.925926,0.074074
"""MMMM""",3,0.666667,0.333333
"""MMMM""",2,0.666667,0.333333


## Unparsable answers:

In [11]:
unparsable_answers_per_model = defaultdict(int)

### In the baseline:

In [12]:
for answer in (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .filter(pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___"))
    .select(["final_answer", "model_name"])
    .iter_rows()
):
    print(answer[1], ": \n")
    print(answer[0][-150:])
    print("-" * 50)
    unparsable_answers_per_model[answer[1]] += 1

Qwen/Qwen3-4B : 

sec². So, if I have h1 - h2 in Btu/lbm, I need to convert it to ft²/sec². Let me use the following conversion:

1 Btu/lbm = 778.169 ft-lbf/lbm

Then, 
--------------------------------------------------
Qwen/Qwen3-4B : 

rsible expansion of an ideal gas, the relation between temperature and pressure is given by $ T_2 = T_1 \left( \frac{P_2}{P_1} \right)^{\frac{\gamma -
--------------------------------------------------
Qwen/Qwen3-0.6B : 

 maybe the velocity coefficient is used with the discharge coefficient. Let me check the formula again.

Alternatively, maybe the velocity coefficient
--------------------------------------------------
Qwen/Qwen3-0.6B : 

1^{2/3}) $

But since $ T_1 = \frac{P_1 V_1}{n R} $, and $ T_2 = \frac{P_2 V_2}{n R} $, we can express $ V_2^{2/3} $ in terms of $ V_1 $:

From the ad
--------------------------------------------------
Qwen/Qwen3-14B : 

tate this exactly. However, Option A says "If it's not the case that both Izzy plays Minecraft an

## In the MAD:

In [13]:
for answer in chain(
    mad_analysis.filter(
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.eval(pl.element().str.starts_with("___"))
        .list.any()
    )
    .select(
        "answers_at_beginning",
        AnalysisColumn.parsed_individual_answers_before.value,
        "model_names",
    )
    .iter_rows(),
    mad_analysis.filter(
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.eval(pl.element().str.starts_with("___"))
        .list.any()
    )
    .select(
        "answers_at_end",
        AnalysisColumn.parsed_individual_answers_after.value,
        "model_names",
    )
    .iter_rows(),
):
    for i, parsed in enumerate(answer[1]):
        if parsed.startswith("___"):
            unparsable_answers_per_model[answer[2][i]] += 1
            print(parsed, f"from {answer[2][i]}: \n")
            print(answer[0][i][-150:])
            print("-" * 50)

___not_parsable___ from Qwen/Qwen3-0.6B: 

decay. Yes, that makes sense. So the answer should be E.
</think>

The tendency for migration to decrease with distance is called (E): distance decay.
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-0.6B: 

nd up. Therefore, the new waist size is 36 inches. 

Looking at the options, option A is 36 inches. So the answer should be A.
</think>

A: 36 inches.
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-0.6B: 

 of products would be N2 : H2 : 2NH. But since the rate constants are given, we need to find the ratio based on the given k_a/k_b/k_c. 

Alternatively
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-0.6B: 

, maybe the current deficit is $18.07 billion, which would be option A. But I can't see why. Alternatively, maybe I made a mistake in the calculation.
--------------------------------------------------
___not_parsable___ f

-> Qwen 0.6B often says "Answer should be A /think. The Answer is G"

In [14]:
unparsable_answers_per_model

defaultdict(int,
            {'Qwen/Qwen3-4B': 376,
             'Qwen/Qwen3-0.6B': 856,
             'Qwen/Qwen3-14B': 317})

### Build an overview of parsing error Influence

In [15]:
baseline_frame_with_group_constellation = baseline_frame.with_columns(
    group_constellation=pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING)
    + pl.lit(" (Baseline)")
).drop("model_name")

parsing_error_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for handling in ["null", "wrong", "random"]:
    new_comparer = AnswerComparer(AnswerOptions.letters_A_to_J, handling)

    new_mad = (
        apply_parsing_and_group_decision(mad_frame, parser, new_comparer, group_reply)
        .with_columns(make_group_constellation())
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    new_base = (
        baseline_frame_with_group_constellation.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            is_correct=new_comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            )
        )
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    parsing_error_influence_table = parsing_error_influence_table.join(
        pl.concat([new_mad, new_base], how="diagonal").select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({handling})"),
        ),
        on="group_constellation",
        how="inner",
    )

parsing_error_influence_table.with_columns(
    deviation=(
        pl.max_horizontal(pl.exclude("group_constellation"))
        - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

group_constellation,Accuracy (null),Accuracy (wrong),Accuracy (random),deviation
str,f64,f64,f64,f64
"""M""",0.597826,0.55,0.57,0.047826
"""LM""",0.616279,0.53,0.54,0.086279
"""HHHM""",0.683673,0.67,0.68,0.013673
"""L""",0.247312,0.23,0.23,0.017312
"""HHHH""",0.721649,0.7,0.71,0.021649
…,…,…,…,…
"""HHM""",0.714286,0.7,0.7,0.014286
"""HM""",0.691489,0.65,0.65,0.041489
"""M (Baseline)""",0.670455,0.59,0.6,0.080455


-> Using (null) is the best, as unparsed values increase your score... should not be used

-> Using wrong is the worst, could possibly be used to be fair, as it is "not correct"

-> Papers and Benchmarks often use "random", which increases the values artificially and introduces noise.. I do not like it, but to be fair one should use it.

---

-> But in all of out cases, even absolute deviation is actually pretty low. (it gets mitigated in group decisions, as unparsable values are ignored in aggregation)

# Agreeableness -> Number of Cases where they reach conslusion

In [16]:
unanimity = (
    mad_analysis.with_columns(
        pl.col(AnalysisColumn.parsed_individual_answers_after.value)
        .list.n_unique()
        .eq(1)
        .alias("End unanimity"),
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.n_unique()
        .eq(1)
        .alias("Start unanimity"),
    )
    .group_by("group_constellation")
    .agg(
        pl.col("End unanimity").mean(),
        pl.col("Start unanimity").mean(),
        pl.when(pl.col("Start unanimity"))
        .then(None)
        .otherwise(pl.col("End unanimity"))
        .mean()
        .alias("End Unanimity | not Start Unanimity"),
        pl.when(pl.col("Start unanimity"))
        .then(pl.col("End unanimity"))
        .otherwise(None)
        .mean()
        .alias("End Unanimity | Start Unanimity"),
        accuracy=pl.col("is_correct").mean(),
    )
)

unanimity

group_constellation,End unanimity,Start unanimity,End Unanimity | not Start Unanimity,End Unanimity | Start Unanimity,accuracy
str,f64,f64,f64,f64,f64
"""HHHM""",0.92,0.64,0.833333,0.96875,0.67
"""LLLL""",0.76,0.3,0.7,0.9,0.26
"""HHHL""",0.7,0.24,0.644737,0.875,0.64
"""LM""",0.79,0.32,0.764706,0.84375,0.53
"""HLMM""",0.82,0.21,0.772152,1.0,0.62
…,…,…,…,…,…
"""HL""",0.72,0.33,0.61194,0.939394,0.53
"""HHHH""",0.98,0.72,0.928571,1.0,0.7
"""HMMM""",0.93,0.72,0.928571,0.930556,0.67


-> In Human Groups (see Group Problem Solving) there is the tendency that the smarter the group, the more it is a "Truth Supported" Decision Scheme, the "dumber" the group, the more it is "proportional"

### Correlation for groups (ignoring 1 member, as it is always 1)

In [17]:
print("Pearson Correlation:")
print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .corr()
)

print("Spearman Rank Correlation:")

print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

Pearson Correlation:
shape: (5, 5)
┌───────────────┬─────────────────┬─────────────────────┬───────────────────────┬──────────┐
│ End unanimity ┆ Start unanimity ┆ End Unanimity | not ┆ End Unanimity | Start ┆ accuracy │
│ ---           ┆ ---             ┆ Start Unan…         ┆ Unanimit…             ┆ ---      │
│ f64           ┆ f64             ┆ ---                 ┆ ---                   ┆ f64      │
│               ┆                 ┆ f64                 ┆ f64                   ┆          │
╞═══════════════╪═════════════════╪═════════════════════╪═══════════════════════╪══════════╡
│ 1.0           ┆ 0.84163         ┆ 0.958687            ┆ 0.537227              ┆ 0.527037 │
│ 0.84163       ┆ 1.0             ┆ 0.745945            ┆ 0.369057              ┆ 0.37374  │
│ 0.958687      ┆ 0.745945        ┆ 1.0                 ┆ 0.423489              ┆ 0.48994  │
│ 0.537227      ┆ 0.369057        ┆ 0.423489            ┆ 1.0                   ┆ 0.513904 │
│ 0.527037      ┆ 0.37374         ┆

-> significantly correlated

---
-> Accuracy Correlates with Unanimity
Of course this could be that if everyone is correct in the beginning, then if they that way, then the chance of being correct is higher,
But can we increase the accuracy by accepting when a group starts with one answer?

## Analysis: Is the group finding answers "together" ? (e.g. can it find answers out of wrong start)

In [18]:
correctness_influence = (
    mad_analysis.with_columns(
        correct_before=comparer(
            pl.col(AnalysisColumn.parsed_combined_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("group_constellation")
    .agg(
        pl.when(pl.col("correct_before"))
        .then(pl.col("is_correct"))
        .otherwise(None)
        .mean()
        .alias("Correct | Correct in Beginning"),
        pl.when(pl.col("correct_before"))
        .then(None)
        .otherwise(pl.col("is_correct"))
        .mean()
        .alias("Correct | !Correct in Beginning"),
        accuracy=pl.col("is_correct").mean(),
    )
)

correctness_influence

group_constellation,Correct | Correct in Beginning,Correct | !Correct in Beginning,accuracy
str,f64,f64,f64
"""HHHH""",0.971429,0.066667,0.7
"""H""",0.986111,0.142857,0.75
"""L""",0.714286,0.101266,0.23
"""LLLL""",0.655172,0.098592,0.26
"""LLL""",0.774194,0.188406,0.37
…,…,…,…
"""LMM""",1.0,0.121951,0.64
"""LMMM""",0.933333,0.125,0.61
"""HHH""",0.972222,0.142857,0.74


While the difference in Correct | Correct in Beginning is negligible, the true power lies in changing the answer when They are not correct.

In [19]:
(
    correctness_influence.join(unanimity, how="left", on="group_constellation")
    .filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

Correct | Correct in Beginning,Correct | !Correct in Beginning,accuracy,End unanimity,Start unanimity,End Unanimity | not Start Unanimity,End Unanimity | Start Unanimity,accuracy_right
f64,f64,f64,f64,f64,f64,f64,f64
1.0,-0.14882,0.76005,0.653338,0.481993,0.681992,0.393411,0.76005
-0.14882,1.0,-0.208889,-0.415312,-0.407628,-0.320593,-0.175008,-0.208889
0.76005,-0.208889,1.0,0.631664,0.437494,0.630809,0.478076,1.0
0.653338,-0.415312,0.631664,1.0,0.866815,0.931826,0.515492,0.631664
0.481993,-0.407628,0.437494,0.866815,1.0,0.754515,0.24823,0.437494
0.681992,-0.320593,0.630809,0.931826,0.754515,1.0,0.378325,0.630809
0.393411,-0.175008,0.478076,0.515492,0.24823,0.378325,1.0,0.478076
0.76005,-0.208889,1.0,0.631664,0.437494,0.630809,0.478076,1.0


## Influence of Group Aggregation Protocol

In [20]:
group_reply_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for strategy in [MajorityVote(), SingularityVote()]:
    new_group_reply_agg = GroupReplyAggregator(strategy)
    group_reply_influence_table = group_reply_influence_table.join(
        apply_parsing_and_group_decision(
            mad_frame, parser, comparer, new_group_reply_agg
        )
        .with_columns(make_group_constellation())
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
        .select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({strategy.__class__.__name__})"),
        ),
        on="group_constellation",
        how="inner",
    )

group_reply_influence_table = group_reply_influence_table.with_columns(
    deviation=(
        pl.max_horizontal(pl.exclude("group_constellation"))
        - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

group_reply_influence_table

group_constellation,Accuracy (MajorityVote),Accuracy (SingularityVote),deviation
str,f64,f64,f64
"""HH""",0.77,0.75,0.02
"""MMM""",0.58,0.57,0.01
"""HHH""",0.74,0.71,0.03
"""HLL""",0.46,0.32,0.14
"""LMM""",0.64,0.59,0.05
…,…,…,…
"""LLM""",0.56,0.49,0.07
"""HL""",0.53,0.5,0.03
"""HHMM""",0.61,0.58,0.03


In [21]:
group_reply_influence_table.drop("group_constellation").with_columns(
    pl.all().rank()
).corr()

Accuracy (MajorityVote),Accuracy (SingularityVote),deviation
f64,f64,f64
1.0,0.939654,-0.052763
0.939654,1.0,-0.293191
-0.052763,-0.293191,1.0


-> Slightly Negative Correlation between Accuracy and the deviation, meaning the better the more MajorityVote == SingularityVote -> Same argument as before

## Inter-Model Correctness Correlation (Aka answer diversity)

In [22]:
mad_analysis

shape: (3_400, 23)
┌───────┬────────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id    ┆ run_id ┆ question_i ┆ phoenix_sp ┆ … ┆ ___parsed_ ┆ ___parsed_ ┆ is_correct ┆ group_cons │
│ ---   ┆ ---    ┆ d          ┆ an_id      ┆   ┆ combined_a ┆ combined_a ┆ ---        ┆ tellation  │
│ i64   ┆ i64    ┆ ---        ┆ ---        ┆   ┆ nswers_bef ┆ nswers_aft ┆ bool       ┆ ---        │
│       ┆        ┆ i64        ┆ str        ┆   ┆ …          ┆ …          ┆            ┆ str        │
│       ┆        ┆            ┆            ┆   ┆ ---        ┆ ---        ┆            ┆            │
│       ┆        ┆            ┆            ┆   ┆ str        ┆ str        ┆            ┆            │
╞═══════╪════════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 3260  ┆ 43     ┆ 5          ┆ 1aa14e4c91 ┆ … ┆ J          ┆ J          ┆ true       ┆ H          │
│       ┆        ┆            ┆ 5cb1a3     ┆   ┆            ┆            ┆            ┆            │
│ 3261  ┆ 43     ┆ 6          ┆ c0ea9a6816 ┆ … ┆ I          ┆ I          ┆ true       ┆ H          │
│       ┆        ┆            ┆ 5ac33e     ┆   ┆            ┆            ┆            ┆            │
│ 3262  ┆ 43     ┆ 0          ┆ 6bc64c3c96 ┆ … ┆ G          ┆ G          ┆ true       ┆ H          │
│       ┆        ┆            ┆ 1beda7     ┆   ┆            ┆            ┆            ┆            │
│ 3263  ┆ 43     ┆ 8          ┆ 35ee808025 ┆ … ┆ C          ┆ C          ┆ false      ┆ H          │
│       ┆        ┆            ┆ 489b27     ┆   ┆            ┆            ┆            ┆            │
│ 3264  ┆ 43     ┆ 2          ┆ ee9c12870a ┆ … ┆ F          ┆ F          ┆ true       ┆ H          │
│       ┆        ┆            ┆ 3b979e     ┆   ┆            ┆            ┆            ┆            │
│ …     ┆ …      ┆ …          ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ 34855 ┆ 352    ┆ 68         ┆ fd114d6a38 ┆ … ┆ I          ┆ I          ┆ true       ┆ HLM        │
│       ┆        ┆            ┆ 99064b     ┆   ┆            ┆            ┆            ┆            │
│ 34856 ┆ 352    ┆ 141        ┆ 30a0911e0e ┆ … ┆ F          ┆ F          ┆ true       ┆ HLM        │
│       ┆        ┆            ┆ fbd5dd     ┆   ┆            ┆            ┆            ┆            │
│ 34857 ┆ 352    ┆ 27         ┆ dc23bc8eef ┆ … ┆ ___all_vot ┆ E          ┆ true       ┆ HLM        │
│       ┆        ┆            ┆ 8f7788     ┆   ┆ es_invalid ┆            ┆            ┆            │
│       ┆        ┆            ┆            ┆   ┆ ___        ┆            ┆            ┆            │
│ 34858 ┆ 352    ┆ 79         ┆ e50f91de53 ┆ … ┆ ___differe ┆ E          ┆ false      ┆ HLM        │
│       ┆        ┆            ┆ b869cb     ┆   ┆ nt_votes__ ┆            ┆            ┆            │
│       ┆        ┆            ┆            ┆   ┆ _          ┆            ┆            ┆            │
│ 34859 ┆ 352    ┆ 142        ┆ 94e15af76b ┆ … ┆ E          ┆ E          ┆ true       ┆ HLM        │
│       ┆        ┆            ┆ a6017c     ┆   ┆            ┆            ┆            ┆            │
└───────┴────────┴────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘

#### For Individual answers

In [23]:
individual_models_answer_per_question = (
    (
        baseline_frame.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING),
            is_correct=comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            ),
        )
        .pivot(
            values="is_correct",
            index="question_id",
            on="model_name",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .select("question_id", "L", "M", "H")
    .with_columns(pl.exclude("question_id").cast(bool))
)

individual_models_answer_per_question

question_id,L,M,H
i64,bool,bool,bool
0,false,true,true
1,false,false,true
2,false,false,true
3,false,true,false
4,false,false,true
…,…,…,…
155,true,true,true
156,true,true,false
157,true,true,true


In [24]:
individual_models_answer_per_question.drop("question_id").corr()

L,M,H
f64,f64,f64
1.0,0.313313,0.2446
0.313313,1.0,0.603185
0.2446,0.603185,1.0


--> Actually surprisingly different

In [25]:
df = pl.concat(
    [
        individual_models_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
)

df_single = df.select("Given", "L", "M", "H")

df_single

Given,L,M,H
str,f64,f64,f64
"""L_incorrect""",0.0,0.476923,0.553846
"""L_correct""",1.0,0.8,0.8
"""M_incorrect""",0.170732,0.0,0.292683
"""M_correct""",0.474576,1.0,0.881356
"""H_incorrect""",0.194444,0.194444,0.0
"""H_correct""",0.4375,0.8125,1.0


-> They are not completely overlapping in the single answer case.

This means could be some diversity effect going on

---

In [26]:
individual_groups_answer_per_question = (
    (
        mad_analysis.filter(pl.col("group_constellation").str.len_chars() == 1).pivot(
            values="is_correct",
            index="question_id",
            on="group_constellation",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .with_columns(pl.exclude("question_id").cast(bool))
)

df_single_mad = pl.concat(
    [
        individual_groups_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", "L", "M", "H")

df_single_mad

Given,L,M,H
str,f64,f64,f64
"""L_incorrect""",0.0,0.454545,0.688312
"""L_correct""",1.0,0.869565,0.956522
"""M_incorrect""",0.066667,0.0,0.466667
"""M_correct""",0.363636,1.0,0.981818
"""H_correct""",0.293333,0.72,1.0
"""H_incorrect""",0.04,0.04,0.0


In [27]:
combined_ind = individual_models_answer_per_question.join(
    individual_groups_answer_per_question, on="question_id", suffix="_mad"
)
pl.concat(
    [
        combined_ind.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in combined_ind.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", pl.exclude("Given"))

Given,L,M,H,H_mad,M_mad,L_mad
str,f64,f64,f64,f64,f64,f64
"""L_correct""",1.0,0.8,0.8,0.857143,0.771429,0.514286
"""L_incorrect""",0.0,0.476923,0.553846,0.692308,0.430769,0.076923
"""M_incorrect""",0.170732,0.0,0.292683,0.487805,0.146341,0.02439
"""M_correct""",0.474576,1.0,0.881356,0.932203,0.830508,0.372881
"""H_correct""",0.4375,0.8125,1.0,0.9375,0.734375,0.3125
…,…,…,…,…,…,…
"""H_mad_correct""",0.4,0.733333,0.8,1.0,0.72,0.293333
"""M_mad_incorrect""",0.177778,0.222222,0.377778,0.466667,0.0,0.066667
"""M_mad_correct""",0.490909,0.890909,0.854545,0.981818,1.0,0.363636


> See Obsidian

#### Is the "nobody is right case" in HHH actually unanimous?

In [28]:
print("Answers where everyone was wrong:")

everyone_wrong = (
    mad_analysis.filter(pl.col("group_constellation") == "HHH")
    .explode(AnalysisColumn.parsed_individual_answers_before.value)
    .with_columns(
        is_individually_correct=comparer(
            pl.col(AnalysisColumn.parsed_individual_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("question_id")
    .agg(
        number_wrong=pl.len(),
        answers=pl.col(AnalysisColumn.parsed_individual_answers_before.value).implode(),
        is_individually_correct=pl.col("is_individually_correct").implode(),
    )
    .filter(pl.col("is_individually_correct").list.eval(pl.element().not_()).list.all())
    .drop("number_wrong", "is_individually_correct")
)

print(everyone_wrong)
unanimous = (
    everyone_wrong["answers"]
    .filter(everyone_wrong["answers"].list.n_unique() == 1)
    .count()
)
print(f"Of that unanimous: {unanimous}")
print("Contentious:")
print(everyone_wrong["answers"].filter(everyone_wrong["answers"].list.n_unique() != 1))

Answers where everyone was wrong:
shape: (23, 2)
┌─────────────┬─────────────────────────────────┐
│ question_id ┆ answers                         │
│ ---         ┆ ---                             │
│ i64         ┆ list[str]                       │
╞═════════════╪═════════════════════════════════╡
│ 15          ┆ ["A", "A", "A"]                 │
│ 125         ┆ ["___not_parsable___", "___not… │
│ 124         ┆ ["A", "A", "A"]                 │
│ 62          ┆ ["C", "D", "C"]                 │
│ 7           ┆ ["A", "A", "A"]                 │
│ …           ┆ …                               │
│ 12          ┆ ["D", "D", "D"]                 │
│ 121         ┆ ["___not_parsable___", "I", "_… │
│ 149         ┆ ["B", "___not_parsable___", "B… │
│ 3           ┆ ["D", "D", "D"]                 │
│ 79          ┆ ["C", "C", "C"]                 │
└─────────────┴─────────────────────────────────┘
Of that unanimous: 15
Contentious:
shape: (8,)
Series: 'answers' [list[str]]
[
	["C", "D", "C"]
	["E"

C# Intra-Group Correctness Correlation (Aka group diversity)

C# Intra-Group Correctness Correlation (Aka group diversity)